# Week 9-2 · DMP-04 — Event-Driven Back-testing & Working with Live Data Streams
**Module:** Data Analysis & Modeling in Python · Instructor: Dr. Yves Hilpisch (The Python Quants).

Part 2 of the OOP lecture. Part 1 built the *theory* (EMH, random walks, a fast **vectorized** back-test). Now
we refactor into **object-oriented**, **event-driven** code — the architecture real trading systems use, where
*the same code* runs the back-test and the live strategy.

> The lecture streams a 10-year EUR/USD file. We reproduce the whole engine on the shipped `eod_prices.csv`
> (NIFTY index, 962 daily bars 2017→2020).

## 1. The reality check — why this is hard work
Before any code, Hilpisch's sobering motivation:
- UK brokers must publish loss rates: **~70–77%** of retail CFD accounts *lose money* (CMC 76.6%, IG 71%, FXCM 69%) — worse than a coin flip, year after year.
- Even **hedge funds**, with hundreds of millions in tech and staff, generally **fail to beat a simple SPY buy-and-hold** over 10–20 years — the *curse of big operations* (costs, compliance, overhead).
- There is no shortcut: **finance + math + programming** are reinforcing layers you must speak fluently.

> The retail edge isn't more resources — it's *low costs* and *small niches* the big players can't be bothered with.

## 2. From functions to classes — encapsulation
OOP lets us **encapsulate** a concept (a price bar, a position, a strategy) into a self-contained object we can
reuse and recombine like **Lego bricks**. We start with the four **event** types — the vocabulary of the engine.
Each event has two defining features: a **timestamp** and a **description of what happened**.

In [1]:
from dataclasses import dataclass, field
import pandas as pd, numpy as np
from collections import deque

@dataclass
class Event:
    time: pd.Timestamp

@dataclass
class MarketEvent(Event):       # a new price bar arrived
    price: float

@dataclass
class SignalEvent(Event):       # strategy says go long / short / flat
    direction: int              # +1 long, -1 short, 0 flat

@dataclass
class OrderEvent(Event):        # portfolio wants to trade a quantity
    quantity: float

@dataclass
class FillEvent(Event):         # execution reports the trade done
    quantity: float
    price: float

print("four event types defined:",
      [c.__name__ for c in (MarketEvent, SignalEvent, OrderEvent, FillEvent)])

four event types defined: ['MarketEvent', 'SignalEvent', 'OrderEvent', 'FillEvent']


## 3. The Lego bricks — the four core components
**DataHandler** streams prices as MarketEvents · **Strategy** turns MarketEvents into SignalEvents ·
**Portfolio** turns signals into OrderEvents and tracks equity · **ExecutionHandler** turns orders into
FillEvents. Each is a class; each *knows how to react to one kind of event*.

In [2]:
class CSVDataHandler:
    """Streams daily prices from a CSV one bar at a time (swap this for a live feed later)."""
    def __init__(self, csv_path, column="Close"):
        s = pd.read_csv(csv_path, index_col=0, parse_dates=True)[column]
        self.bars = list(s.items())     # [(timestamp, price), ...]
        self.i = 0
    def stream_next(self):
        if self.i >= len(self.bars):
            return None
        t, p = self.bars[self.i]; self.i += 1
        return MarketEvent(time=t, price=float(p))

print("CSVDataHandler defined")

CSVDataHandler defined


In [3]:
class MomentumStrategy:
    """Time-series momentum: if recent return is positive -> go long, else short.
       lags = how many days of momentum; threshold = ignore tiny moves (a hyper-parameter)."""
    def __init__(self, lags=1, threshold=0.0):
        self.lags = lags
        self.threshold = threshold
        self.prices = []
    def on_market_event(self, ev, queue):
        self.prices.append(ev.price)
        if len(self.prices) <= self.lags:
            return
        # momentum = mean of the last `lags` daily log returns
        arr = np.array(self.prices)
        rets = np.log(arr[-(self.lags+1):] / np.roll(arr[-(self.lags+1):], 1))[1:]
        mom = rets.mean()
        if abs(mom) < self.threshold:
            return                              # below threshold -> no new signal
        direction = 1 if mom > 0 else -1
        queue.append(SignalEvent(time=ev.time, direction=direction))

print("MomentumStrategy defined")

MomentumStrategy defined


In [4]:
class SimplePortfolio:
    """Tracks position, cash and equity for a single instrument."""
    def __init__(self, start_cash=100_000):
        self.cash = start_cash
        self.position = 0          # units held (+ long, - short)
        self.last_price = None
        self.target = 0            # desired position from latest signal
        self.equity_curve = []
    def on_market_event(self, ev, queue):
        self.last_price = ev.price
        equity = self.cash + self.position * ev.price      # mark-to-market
        self.equity_curve.append((ev.time, equity))
    def on_signal_event(self, ev, queue):
        # go ALL-IN long or short: size the position to the full equity (single instrument)
        equity = self.cash + self.position * self.last_price
        target_units = ev.direction * (equity / self.last_price)
        qty = target_units - self.position                  # trade to reach target
        if qty != 0:
            queue.append(OrderEvent(time=ev.time, quantity=qty))
    def on_fill_event(self, ev, queue):
        self.position += ev.quantity
        self.cash -= ev.quantity * ev.price                 # pay (or receive) cash

class NaiveExecutionHandler:
    """Assume every order fills immediately at the last price."""
    def __init__(self, portfolio):
        self.portfolio = portfolio
    def on_order_event(self, ev, queue):
        queue.append(FillEvent(time=ev.time, quantity=ev.quantity,
                               price=self.portfolio.last_price))

print("SimplePortfolio + NaiveExecutionHandler defined")

SimplePortfolio + NaiveExecutionHandler defined


## 4. The engine — a single event queue and an event loop
A central **queue** (a `deque`) holds events. The loop pops the next event, routes it to the components by
**type**, and they may append *new* events. This is the skeleton of a live trading system — only the
DataHandler differs between back-test and production.

In [5]:
class BacktestEngine:
    def __init__(self, data, strategy, portfolio, execution):
        self.data = data; self.strategy = strategy
        self.portfolio = portfolio; self.execution = execution
        self.queue = deque()
    def run(self):
        while True:
            market = self.data.stream_next()      # pull next bar
            if market is None:
                break
            self.queue.append(market)
            while self.queue:                     # drain all events from this bar
                ev = self.queue.popleft()
                if isinstance(ev, MarketEvent):
                    self.strategy.on_market_event(ev, self.queue)
                    self.portfolio.on_market_event(ev, self.queue)
                elif isinstance(ev, SignalEvent):
                    self.portfolio.on_signal_event(ev, self.queue)
                elif isinstance(ev, OrderEvent):
                    self.execution.on_order_event(ev, self.queue)
                elif isinstance(ev, FillEvent):
                    self.portfolio.on_fill_event(ev, self.queue)
        return pd.Series(dict(self.portfolio.equity_curve))

print("BacktestEngine defined")

BacktestEngine defined


## 5. Run the event-driven back-test (1-day momentum)

In [6]:
def run_strategy(lags, threshold):
    data = CSVDataHandler("eod_prices.csv")
    strat = MomentumStrategy(lags=lags, threshold=threshold)
    port  = SimplePortfolio(start_cash=100_000)
    exe   = NaiveExecutionHandler(port)
    eq = BacktestEngine(data, strat, port, exe).run()
    return eq

equity = run_strategy(lags=1, threshold=0.0)
# normalize to growth-of-1 and compare to buy & hold
px = pd.read_csv("eod_prices.csv", index_col=0, parse_dates=True)["Close"]
bh = px / px.iloc[0]
strat_curve = equity / equity.iloc[0]
print(f"bars processed        : {len(equity)}")
print(f"event-momentum total  : {(strat_curve.iloc[-1]-1)*100:6.2f}%")
print(f"buy & hold total      : {(bh.iloc[-1]-1)*100:6.2f}%")

bars processed        : 962
event-momentum total  : -10.47%
buy & hold total      :  58.55%


## 6. The hyper-parameters: `lags` and `threshold`
Two knobs drive the strategy. **`lags`** = how many days of momentum (AR(1) vs longer). **`threshold`** ignores
tiny moves to cut needless turnover — Hilpisch calls it as important as the strategy idea itself, a true
*hyper-parameter*. Let's sweep both.

In [7]:
rows = []
for lags in (1, 3, 5, 7):
    for thr in (0.0, 0.005, 0.01):
        eq = run_strategy(lags, thr)
        tot = eq.iloc[-1]/eq.iloc[0] - 1
        rows.append((lags, thr, round(tot*100, 2)))
sweep = pd.DataFrame(rows, columns=["lags", "threshold", "total_return_%"])
print(sweep.to_string(index=False))
print(f"\nbuy & hold total: {(bh.iloc[-1]-1)*100:.2f}%")

 lags  threshold  total_return_%
    1      0.000          -10.47
    1      0.005            0.32
    1      0.010           -0.13
    3      0.000           15.36
    3      0.005           50.64
    3      0.010           61.40
    5      0.000           32.79
    5      0.005          -14.92
    5      0.010           50.36
    7      0.000           45.65
    7      0.005           10.40
    7      0.010           85.81

buy & hold total: 58.55%


## 7. Risk stats — momentum vs buy & hold
Annualize with √252. Note: a long/short strategy on a *single* instrument has the **same volatility** as the
underlying (volatility doesn't care about the sign), so the honest comparison is total return *and* drawdown.

In [8]:
def stats(curve):
    r = np.log(curve/curve.shift(1)).dropna()
    ann_r = r.mean()*252; ann_v = r.std()*np.sqrt(252)
    sharpe = ann_r/ann_v if ann_v else np.nan
    dd = (curve/curve.cummax() - 1).min()
    return ann_r, ann_v, sharpe, dd

for name, c in [("Buy & Hold", bh), ("Event momentum (1-lag)", strat_curve)]:
    ar, av, sh, dd = stats(c)
    print(f"{name:24s} annRet {ar*100:6.2f}%  annVol {av*100:6.2f}%  Sharpe {sh:5.2f}  maxDD {dd*100:6.2f}%")

Buy & Hold               annRet  12.09%  annVol  19.21%  Sharpe  0.63  maxDD -38.44%
Event momentum (1-lag)   annRet  -2.90%  annVol  19.34%  Sharpe -0.15  maxDD -43.03%


## 8. Equity curve chart

In [9]:
import matplotlib
matplotlib.use("Agg")
import matplotlib.pyplot as plt
fig, ax = plt.subplots(figsize=(8,3.6))
ax.plot(bh.index, bh, color="#64748b", lw=1.2, ls="--",
        label=f"Buy & Hold ({(bh.iloc[-1]-1)*100:.0f}%)")
ax.plot(strat_curve.index, strat_curve, color="#0284c7", lw=1.5,
        label=f"Event-driven momentum ({(strat_curve.iloc[-1]-1)*100:.0f}%)")
ax.axhline(1, color="gray", lw=0.6)
ax.set_title("Event-driven momentum back-test (NIFTY daily)")
ax.set_ylabel("Growth of 1"); ax.legend(); ax.grid(alpha=0.25)
fig.tight_layout(); fig.savefig("preview.png", dpi=110)
print("equity curve drawn")

equity curve drawn


## 9. From back-test to live data streams
The pay-off of this architecture: to go **live**, you swap *one brick*. The `CSVDataHandler` "streams" by
reading a file; a live handler streams by listening to a broker's websocket / REST feed (crypto markets stream
all day). The Strategy, Portfolio and Execution classes — and the event loop — stay **identical**. That is why
professionals insist the back-test engine and the live engine are *the same code*, which removes the friction
(and bugs) of porting research to production.

```python
class LiveDataHandler:               # same interface as CSVDataHandler
    def stream_next(self):
        tick = broker.get_next_tick()   # websocket / REST
        return MarketEvent(time=tick.time, price=tick.price)
# BacktestEngine(LiveDataHandler(), strategy, portfolio, execution).run()  # live!
```

## Key takeaways
- **Reality:** ~70–77% of retail accounts lose money; hedge funds rarely beat SPY long-term. No shortcuts — finance + math + programming are one skill set.
- **Encapsulation:** wrap concepts (events, data, strategy, portfolio) into classes you reuse like Lego bricks.
- **Event-driven > vectorized** when you need order queues, partial fills, latency, path-dependence, multi-asset — it models *streams of events*, not whole arrays.
- **Four events** (Market → Signal → Order → Fill), each a `dataclass` with a timestamp + description, flow through a single **queue**; the **event loop** routes each by type.
- **Four bricks:** DataHandler, Strategy, Portfolio, ExecutionHandler — the same skeleton as a live system.
- **Momentum** = time-series momentum (sign of recent return), tied to autocorrelation; **`lags`** and **`threshold`** are the hyper-parameters (threshold cuts turnover and matters as much as the idea).
- **Single-instrument long/short** keeps the underlying's volatility — judge on return *and* drawdown (watch multi-year drawdown durations).
- **Going live = swap one brick:** replace the CSV DataHandler with a streaming feed; everything else is unchanged. Back-test code == live code.